# API Rate Limiter

*1991 Academy · Missions*

Your API allows `max_calls` requests per `window_ms` milliseconds per client — the guard that turns abuse into a polite `429`.

Write `create_limiter(max_calls, window_ms)` returning a function `allow(t)` (t = timestamp in ms). It returns `True` if fewer than `max_calls` *allowed* calls happened in `(t - window_ms, t]`, else `False`. Denied calls don't count against the window. A queue of timestamps — evict the expired ones from the front — is the classic sliding-window implementation.

> 💾 **Your work is saved:** uploaded to Colab (**File → Upload notebook**), this notebook lives in your Google Drive, in the *Colab Notebooks* folder. Close it any time and continue later.
>
> ✅ When every test passes, go back to 1991 Academy and click **I solved it ✓**.

## Your code

Write your solution in the cell below and run it: click ▶, or press Shift+Enter.

In [ ]:
# Sliding window rate limiter.
# allow(t): True if fewer than max_calls ALLOWED calls
# happened in the window (t - window_ms, t].
def create_limiter(max_calls, window_ms):
    # keep allowed timestamps in a queue
    def allow(t):
        return True
    return allow

## Check your solution

Run the cell below. If a test fails, fix your code, run your code cell again, then this one.

In [ ]:
#@title ▶ Run the tests { display-mode: "form" }
import json, math, traceback
MSG = {
 "all_pass": "🎉 All {n} tests pass! Go back to 1991 Academy and click “I solved it ✓”.",
 "some_pass": "{p}/{n} tests pass. Keep going!",
 "none_ran": "No tests ran. Is your function named {fn}?",
 "expected": "expected {e}, got {a}",
 "error": "💥 The tests stopped with an error:",
 "not_run": "Did you run your code cell above? The tests only see code that has been run."
}

def _report(results, error, fn):
    for name, ok, expected, actual in results:
        print(("✓ " if ok else "✗ ") + name + ("" if ok else "   " + MSG["expected"].format(e=expected, a=actual)))
    if error:
        print("\n" + MSG["error"])
        print(error.rstrip())
    passed = sum(1 for r in results if r[1])
    print()
    if results and passed == len(results) and not error:
        print(MSG["all_pass"].format(n=passed))
    elif results:
        print(MSG["some_pass"].format(p=passed, n=len(results)))
    elif not error:
        print(MSG["none_ran"].format(fn=fn))

def _academy_run_tests():
    results = []

    def __check(name, actual, expected):
        try:
            ok = bool(actual == expected)
        except Exception:
            ok = False
        results.append((str(name), ok, repr(expected), repr(actual)))

    def tests():
        allow = create_limiter(3, 1000)
        __check("first three pass", [allow(0), allow(100), allow(200)], [True, True, True])
        __check("fourth inside window blocked", allow(300), False)
        __check("still blocked at window edge", allow(999), False)
        __check("oldest expires, allowed again", allow(1001), True)
        a2 = create_limiter(1, 100)
        __check("independent limiter state", a2(0), True)
        __check("blocked inside its window", a2(50), False)
        __check("free after the window passes", a2(151), True)

    error = None
    try:
        tests()
    except Exception as exc:
        error = traceback.format_exc(limit=-3)
        if isinstance(exc, NameError):
            error += "\n" + MSG["not_run"]
    _report(results, error, 'create_limiter')

_academy_run_tests()

## Hints

<details><summary>Hint 1</summary>

Keep the allowed timestamps in a queue. On each allow(t): first drop the timestamps <= t - window_ms from the front, then compare its length with max_calls.

</details>

<details><summary>Hint 2</summary>

Only add t when the call is allowed — denied requests never join the queue.

</details>

<details><summary>Hint 3</summary>

A collections.deque created in create_limiter, which allow() reads and changes: popleft() the expired timestamps, append(t) when you allow. Each limiter gets its own deque.

</details>